# Q2 — Người thuê đang “trả tiền” cho **nội thất** hay cho yếu tố **near** nhiều hơn? (EDA)

**Near** chỉ gồm: `near_school`, `near_market`, `near_supermarket`.

## 1. Câu hỏi nghiên cứu
**Câu hỏi:** *“Trong từng khu vực (district), ‘premium’ (mức đội giá) do **near** và do **nội thất/tiện nghi trong phòng** cái nào lớn hơn? Near premium và furnishing premium khác nhau như thế nào giữa các district?”*

**Tính xác thực:** Có thể trả lời trực tiếp bằng dữ liệu thông qua:
- Xây dựng **Near Index** từ 3 biến near (`near_school`, `near_market`, `near_supermarket`)
- Xây dựng **Furnishing Index** từ các biến nội thất/tiện nghi hiện có
- So sánh **phân phối `price_per_sqm`** giữa các mức *Low/Mid/High* của từng index, theo từng district (EDA, không dùng model).


## 2. Động cơ & Lợi ích
* **Vì sao đáng nghiên cứu?** Người thuê thường phải chọn: *“gần trường/chợ/siêu thị”* hay *“đủ nội thất”*.  
  Nếu biết district nào **đắt vì near** và district nào **đắt vì nội thất**, ta sẽ tối ưu tìm kiếm và thương lượng tốt hơn.
* **Lợi ích & Insight:**
  - Xếp hạng district theo **Near premium** và **Furnishing premium**
  - Tìm “điểm ngọt”: **near cao nhưng nội thất vừa phải** (hoặc ngược lại) để tối ưu giá/m²
* **Hỗ trợ thực tế:** Với cùng ngân sách, biết nên **đánh đổi** cái gì ở mỗi district.


### A. Tiền xử lý dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PATH = "/mnt/data/data_clean.csv"
df = pd.read_csv(PATH)

# Numeric safety
for col in ["price","area","price_per_sqm","amenity_score","location_score","security_score","log_price",
            "near_school","near_market","near_supermarket"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

def normalize_district(x):
    if pd.isna(x):
        return x
    s = str(x).strip().lower()
    if s.isdigit():
        return f"q{s}"
    return " ".join([w.capitalize() for w in s.split()])

df["district_std"] = df["district"].apply(normalize_district)
df.shape

In [ ]:
# Sanity check: required near columns
required_near = ["near_school","near_market","near_supermarket"]
missing_near = [c for c in required_near if c not in df.columns]
missing_near

### B. Phân tích
#### B1) Xây dựng Near Index (chỉ 3 biến near) và Furnishing Index (nội thất/tiện nghi)

- **Near Index** = trung bình của `near_school`, `near_market`, `near_supermarket`
- **Furnishing Index** = trung bình của các đặc trưng nội thất/tiện nghi trong phòng (dạng nhị phân)  
  + Với biến phân loại (ví dụ `washing_machine_type`) ta quy đổi thành 0/1 dựa trên việc “có/không”.


In [ ]:
# Keep only valid rows for price_per_sqm analysis
work = df.copy()

# Filter non-positive / missing essentials
work = work[(work["price_per_sqm"].notna()) & (work["price_per_sqm"] > 0) & (work["area"] > 0) & (work["price"] > 0)].copy()

# --- Near Index (ONLY 3 near columns) ---
for c in required_near:
    work[c] = work[c].fillna(0)  # assume missing => not near
work["near_index"] = work[required_near].mean(axis=1)

# --- Furnishing features (binary-ish) ---
binary_furnish_cols = [
    "has_air_conditioner","has_kitchen","has_fridge","has_wardrobe","has_bed","has_mattress","has_sofa",
    "is_full_furniture","is_duplex","has_window","has_balcony","has_wifi","has_parking","has_private_facilities",
    "has_camera","has_security"
]
binary_furnish_cols = [c for c in binary_furnish_cols if c in work.columns]

# Convert to 0/1 safely
for c in binary_furnish_cols:
    work[c] = pd.to_numeric(work[c], errors="coerce").fillna(0).clip(0,1)

# Categorical -> binary: washing_machine_type (has any type)
if "washing_machine_type" in work.columns:
    wm = work["washing_machine_type"].astype(str).str.lower().str.strip()
    # treat common "none"/"khong"/"nan" as 0, else 1
    work["has_washing_machine"] = (~wm.isin(["none","no","khong","không","nan","na","n/a","0",""])).astype(int)
    furnish_cols = binary_furnish_cols + ["has_washing_machine"]
else:
    furnish_cols = binary_furnish_cols

work["furnish_index"] = work[furnish_cols].mean(axis=1)

work[["near_index","furnish_index","price_per_sqm"]].describe().T

#### B2) Chia mức Low/Mid/High theo quantile (tertiles)

Ta chia `near_index` và `furnish_index` thành 3 mức:
- **Low**: dưới Q33
- **Mid**: Q33–Q66
- **High**: trên Q66


In [ ]:
def tertile_label(s):
    q1 = s.quantile(1/3)
    q2 = s.quantile(2/3)
    return pd.cut(s, bins=[-np.inf, q1, q2, np.inf], labels=["Low","Mid","High"])

work["near_level"] = tertile_label(work["near_index"])
work["furnish_level"] = tertile_label(work["furnish_index"])

work[["near_index","near_level","furnish_index","furnish_level"]].head(10)

#### B3) So sánh premium toàn thị trường (không theo district)

- **Near premium** = median(price/m² | Near=High) − median(price/m² | Near=Low)  
- **Furnishing premium** = median(price/m² | Furnish=High) − median(price/m² | Furnish=Low)


In [ ]:
def median_ppsm_by_level(level_col):
    return work.groupby(level_col)["price_per_sqm"].median().reindex(["Low","Mid","High"])

near_med = median_ppsm_by_level("near_level")
furn_med = median_ppsm_by_level("furnish_level")

near_premium = near_med["High"] - near_med["Low"]
furn_premium = furn_med["High"] - furn_med["Low"]

near_med, furn_med, near_premium, furn_premium

In [ ]:
# Plot: overall premiums
plt.figure(figsize=(7,4))
plt.plot(["Low","Mid","High"], near_med.values, marker="o", label="Near level")
plt.plot(["Low","Mid","High"], furn_med.values, marker="o", label="Furnishing level")
plt.ylabel("Median price per sqm (VND/sqm)")
plt.title("Overall: price/sqm by Near level vs Furnishing level")
plt.legend()
plt.tight_layout()
plt.show()

#### B4) Premium theo district + xếp hạng

Chỉ lấy district có đủ mẫu (`n>=300`) để so sánh ổn định.


In [ ]:
min_n = 300
district_n = work["district_std"].value_counts()
eligible = district_n[district_n >= min_n].index.tolist()

def district_premium(level_col):
    rows = []
    for d in eligible:
        g = work[work["district_std"]==d]
        med = g.groupby(level_col)["price_per_sqm"].median()
        if ("Low" not in med.index) or ("High" not in med.index):
            continue
        rows.append([d, len(g), float(med.get("Low", np.nan)), float(med.get("High", np.nan)), float(med["High"]-med["Low"])])
    return pd.DataFrame(rows, columns=["district","n","ppsm_low","ppsm_high","premium"])

near_prem_d = district_premium("near_level")
furn_prem_d = district_premium("furnish_level")

# Merge
prem = pd.merge(near_prem_d, furn_prem_d, on=["district"], suffixes=("_near","_furn"))
prem["near_minus_furn"] = prem["premium_near"] - prem["premium_furn"]

prem.sort_values("near_minus_furn", ascending=False).head(15)

In [ ]:
# Plot: compare premiums by district (top 15 absolute)
prem2 = prem.copy()
prem2["abs_gap"] = (prem2["premium_near"] - prem2["premium_furn"]).abs()
top15 = prem2.sort_values("abs_gap", ascending=False).head(15)

x = np.arange(len(top15))
w = 0.38

plt.figure(figsize=(10,4))
plt.bar(x - w/2, top15["premium_near"], width=w, label="Near premium (High-Low)")
plt.bar(x + w/2, top15["premium_furn"], width=w, label="Furnishing premium (High-Low)")
plt.xticks(x, top15["district"].astype(str), rotation=45, ha="right")
plt.ylabel("Premium in price/sqm (VND/sqm)")
plt.title("District comparison — Near premium vs Furnishing premium (Top 15 by |gap|)")
plt.legend()
plt.tight_layout()
plt.show()

#### B5) Heatmap 2D: Near level × Furnishing level → median price/m² (toàn thị trường)

Heatmap giúp nhìn nhanh tổ hợp nào “đắt nhất”:
- Near High & Furnish High thường là góc giá cao
- Quan sát xem Near High nhưng Furnish Low có “đắt” nhiều không (premium chủ yếu do near)


In [ ]:
pivot = (work.pivot_table(index="near_level", columns="furnish_level", values="price_per_sqm", aggfunc="median")
         .reindex(index=["Low","Mid","High"], columns=["Low","Mid","High"]))

plt.figure(figsize=(6,4))
plt.imshow(pivot.values, aspect="auto")
plt.xticks(np.arange(3), ["Low","Mid","High"])
plt.yticks(np.arange(3), ["Low","Mid","High"])
plt.colorbar(label="Median price/sqm (VND/sqm)")
plt.xlabel("Furnishing level")
plt.ylabel("Near level")
plt.title("Overall heatmap — Near × Furnishing → median price/sqm")
for i in range(3):
    for j in range(3):
        val = pivot.values[i,j]
        if np.isfinite(val):
            plt.text(j, i, f"{val:,.0f}".replace(",", "."), ha="center", va="center", fontsize=9)
plt.tight_layout()
plt.show()

pivot

## C. KẾT QUẢ VÀ DIỄN GIẢI




### 1) Toàn thị trường: **nội thất “đội giá” rõ hơn near**
- Median **giá/m²** theo **Near level**:  
  **Low ~ 152.000** → Mid ~ 146.667 → **High ~ 144.000 (VND/m²)**  
  ⟹ **Near premium (High − Low) ≈ −8.000 VND/m²** *(không tăng; dữ liệu cho thấy giảm nhẹ)*.
- Median **giá/m²** theo **Furnishing level**:  
  **Low ~ 147.059** → Mid ~ 150.000 → **High ~ 154.286 (VND/m²)**  
  ⟹ **Furnishing premium (High − Low) ≈ +7.227 VND/m²**.

**Diễn giải:** Với 3 biến near (`near_school`, `near_market`, `near_supermarket`) thì **near không tạo premium giá/m² ở mức toàn thị trường**, trong khi **nội thất/tiện nghi trong phòng** có xu hướng làm **giá/m² tăng**.

---

### 2) Theo tổ hợp Near × Furnishing: near không làm giá/m² tăng
Median `price_per_sqm` (VND/m²):
- **Near Low**: Furnish Low ~ **150.000** | Mid ~ **150.000** | High ~ **160.000** (VND/m²)
- **Near Mid**: Furnish Low ~ **144.444** | Mid ~ **145.000** | High ~ **150.000** (VND/m²)
- **Near High**: Furnish Low ~ **140.000** | Mid ~ **144.000** | High ~ **150.000** (VND/m²)

**Diễn giải:** Ở mọi mức nội thất, **near cao không đẩy giá/m² lên**; xu hướng premium chủ yếu đến từ **furnishing**.

---

### 3) Theo district (n ≥ 300): nơi nào “đắt vì near”
Top district có **Near premium** dương nổi bật:
- **q11**: ~ **+17.583 VND/m²**
- **q10**: ~ **+10.714 VND/m²**
- **q7**: ~ **+1.143 VND/m²**
- **q2**: ~ **+714 VND/m²**
- **Bình Tân**: ~ **+667 VND/m²**
- **q4**: ~ **−2.500 VND/m²**

**Diễn giải:** Near premium **chỉ rõ rệt ở một vài district**; nhiều district còn lại near premium nhỏ hoặc âm → “gần” (theo 3 biến near này) **không đảm bảo đắt hơn**.

---

### 4) Theo district (n ≥ 300): nơi nào “đắt vì nội thất”
Top district có **Furnishing premium** cao:
- **q2**: ~ **+23.913 VND/m²**
- **q3**: ~ **+13.617 VND/m²**
- **Thủ Đức**: ~ **+12.857 VND/m²**
- **q10**: ~ **+11.619 VND/m²**
- **Tân Bình**: ~ **+10.000 VND/m²**
- **q11**: ~ **+10.000 VND/m²**

**Diễn giải:** Furnishing premium thường **rõ ràng và nhất quán hơn** near premium trong dữ liệu này.

---

## Kết luận hành động cho người thuê
1) Nếu mục tiêu là **giảm giá/m²**, dữ liệu gợi ý **đừng trả thêm chỉ vì near** (3 biến near này), vì overall near **không tạo premium**.  
2) Nếu cần “đáng sống hơn”, **nội thất/tiện nghi** là yếu tố thường làm giá/m² tăng — hãy cân nhắc trade-off (bớt nội thất để tối ưu giá).  
3) Nếu bạn thuê ở district có near premium cao (ví dụ **q11, q10**), near có thể “đắt” hơn rõ rệt → cân nhắc đánh đổi: near thấp hơn một chút nhưng **diện tích/nội thất** tốt hơn.
